# Golden-case regression curation

Runs the pipeline once on a page, then for each of the three stages that
keep/drop bbox'd items -- **vector classification**, **FAST detection**,
**word-splitting (Radon)** -- shows one *combined, randomised* pool of
candidate clusters/segmentations. You pick any candidates and assign each a
`positive`/`negative` label yourself: a candidate the pipeline currently
keeps can be labelled `negative` (a known false positive), one it drops can
be labelled `positive` (a known miss). What the pipeline actually decided is
stored separately (`role`) and is what the regression check compares.

Picks are saved to `outputs/regression_cases/cases.json`. The **View / delete
cases** cells list and prune the bank; the final **Replay** section re-runs
the pipeline against every stored case (from any prior session too) and
reports PASS/FAIL -- the same cell to re-run later as a regression check.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd() / "golden_case_curation.ipynb").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rastervec.logging_setup import configure_logging
from rastervec.pipelines.current import run_pipeline
from rastervec.paths import output_dir
from rastervec.renderer.notebook import show_row
from rastervec.Evaluation.Evaluate.golden_schema import (
    GoldenCaseBank,
    add_case,
    drop_cases,
    load_cases,
    save_cases,
)
from rastervec.Evaluation.Evaluate.golden_regression import (
    capture_case,
    format_case_bank,
    format_regression_report,
    list_stage_candidates,
    run_regression,
)

configure_logging()

CASES_PATH = str(output_dir("regression_cases") / "cases.json")


def capture_picks(cands, picks, *, stage, bank):
    """picks: {candidate_index: "positive" | "negative"}."""
    for idx, label in picks.items():
        bank = add_case(bank, capture_case(
            cands[idx], stage=stage, label=label,
            pdf_path=str(PDF_PATH), page_index=PAGE_INDEX,
        ))
    save_cases(bank, CASES_PATH)
    print(f"saved {len(bank.cases)} case(s) to {CASES_PATH}")
    return bank


## Parameters

In [ ]:
PDF_PATH = next(iter(sorted((PROJECT_ROOT / "references").glob("*.pdf"))), None)
PAGE_INDEX = 3

assert PDF_PATH is not None, "no PDF under references/ -- set PDF_PATH by hand"
print("PDF:", PDF_PATH, "| page", PAGE_INDEX)

## Run the pipeline

In [ ]:
res = run_pipeline(str(PDF_PATH), PAGE_INDEX, enable_fast=True, verbose=True)

try:
    bank = load_cases(CASES_PATH)
except FileNotFoundError:
    bank = GoldenCaseBank()
print(f"loaded {len(bank.cases)} existing case(s) from {CASES_PATH}")

## 1. Vector Classification

One combined, randomised pool: the final surviving ("kept") text-candidate
clusters **and** every `role="dropped"` group across the 12-step chain and
every (layer, color) bucket, interleaved. Each row prints its structural
`role`; you assign the `positive`/`negative` label yourself in the next cell.

In [ ]:
CLASSIFY_N = 12          # max candidates to show (None = all)
CLASSIFY_SEED = 0        # fixed = repeatable order; None = reshuffled each run

classify_cands = list_stage_candidates(
    res, "classification", n=CLASSIFY_N, shuffle=True, seed=CLASSIFY_SEED,
)
for i, c in enumerate(classify_cands):
    print(f"  [{i}] role={(c.role or '-'):<8} {c.note}  bbox={tuple(round(v, 1) for v in c.bbox)}")
if classify_cands:
    show_row([c.image for c in classify_cands], [f"[{i}] {c.role}" for i, c in enumerate(classify_cands)])

In [ ]:
# Map candidate index -> the label YOU assign it. Any candidate can take
# either label regardless of its printed role: label a "kept" candidate
# "negative" to record a false positive, a "dropped" one "positive" to
# record a miss.
CLASSIFY_PICKS = {
    0: "positive",
    1: "negative",
}

In [ ]:
bank = capture_picks(classify_cands, CLASSIFY_PICKS, stage="classification", bank=bank)

## 2. FAST Detection

One combined, randomised pool of clusters FAST passed **and** dropped. Row
`role` is `passed` / `dropped`; you assign the label yourself.

In [ ]:
FAST_N = 12
FAST_SEED = 0

fast_cands = list_stage_candidates(res, "fast", n=FAST_N, shuffle=True, seed=FAST_SEED)
for i, c in enumerate(fast_cands):
    print(f"  [{i}] role={(c.role or '-'):<8} {c.note}  bbox={tuple(round(v, 1) for v in c.bbox)}")
if fast_cands:
    show_row([c.image for c in fast_cands], [f"[{i}] {c.role}" for i, c in enumerate(fast_cands)])

In [ ]:
# {candidate_index: "positive" | "negative"} -- label freely, see cell 1's note.
FAST_PICKS = {
    0: "positive",
    1: "negative",
}

In [ ]:
bank = capture_picks(fast_cands, FAST_PICKS, stage="fast", bank=bank)

## 3. Word Splitting (Radon)

Word-splitting has no structural pass/drop signal, so every candidate's
`role` is `-`. One randomised pool of segmented clusters (word boxes drawn
in red); `positive`/`negative` is purely your visual judgement of whether
the split looks right.

In [ ]:
WORD_SPLIT_N = 12
WORD_SPLIT_SEED = 0

word_split_cands = list_stage_candidates(
    res, "word_split", n=WORD_SPLIT_N, shuffle=True, seed=WORD_SPLIT_SEED,
)
for i, c in enumerate(word_split_cands):
    print(f"  [{i}] {c.note}  bbox={tuple(round(v, 1) for v in c.bbox)}")
if word_split_cands:
    show_row([c.image for c in word_split_cands], [f"[{i}]" for i in range(len(word_split_cands))])

In [ ]:
# {candidate_index: "positive" | "negative"} -- your visual verdict on each split.
WORD_SPLIT_PICKS = {
    0: "positive",
    1: "negative",
}

In [ ]:
bank = capture_picks(word_split_cands, WORD_SPLIT_PICKS, stage="word_split", bank=bank)

## View all stored cases

Every case in the bank (from this and any prior session). The `[i]` index is
what the delete cell below takes.

In [ ]:
bank = load_cases(CASES_PATH)
print(format_case_bank(bank))

## Delete cases

Set `DELETE_INDICES` to the `[i]` indices from the listing above, then run.

In [ ]:
DELETE_INDICES = []

if DELETE_INDICES:
    before = load_cases(CASES_PATH)
    bank = drop_cases(before, DELETE_INDICES)
    save_cases(bank, CASES_PATH)
    print(f"deleted {len(before.cases) - len(bank.cases)} case(s); {len(bank.cases)} remain\n")
    print(format_case_bank(bank))
else:
    print("set DELETE_INDICES (from the listing above) to delete")

## Replay

Re-runs the pipeline against every case in the bank (from any prior curation
session too -- not just this one) and reports PASS/FAIL. This is the cell a
contributor re-runs later to check for regressions.

In [ ]:
bank = load_cases(CASES_PATH)
results = run_regression(bank)
print(format_regression_report(results))